# Teams Solo QoE Report

In [ ]:
from pathlib import Path
import json
import os
import math
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import sys
REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.git').exists())
sys.path.insert(0, str(REPO_ROOT / 'experiments/designated_experiments'))
from graph_output import install_graph_saver

CAPACITIES = [3, 6, 10]

result_candidates = [
    Path('teams_only_experiment'),
    REPO_ROOT / 'experiments/teams_only_experiment',
]
BASE_DIR = next((p.resolve() for p in result_candidates if p.is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Could not locate teams_only_experiment')

def load_qoe(tier_dir):
    path = tier_dir / 'teams_stats.jsonl'
    if not path.is_file():
        raise FileNotFoundError(f'Missing required QoE data: {path}')
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

tiers = {}
for capacity_mbps in CAPACITIES:
    tier_dir = BASE_DIR / f'{capacity_mbps}mbps'
    qoe = load_qoe(tier_dir)
    start_time = qoe[0]['timestamp']
    seconds = [row['timestamp'] - start_time for row in qoe]
    tiers[capacity_mbps] = {
        'dir': tier_dir,
        'qoe': qoe,
        'stats': [row['stats'] for row in qoe],
        'seconds': seconds,
    }

plt.style.use('seaborn-v0_8-whitegrid')
METRIC_PREFIXES = [
    'download_throughput',
    'upload_throughput',
    'buffer_health',
    'cumulative_dropped_frame_rate',
    'video_resolution_over_time',
    'total_video_frames',
]
GRAPH_NAMES = [
    f'{i + 1:02d}_{metric}_teams_{capacity_mbps}mbps'
    for i, metric in enumerate(METRIC_PREFIXES)
    for capacity_mbps in CAPACITIES
]
GRAPHS_DIR = install_graph_saver(plt, BASE_DIR, GRAPH_NAMES)

TEAMS_COLOR = '#5059C9'

DROP_YMAX = max(
    max(
        100 * (s.get('frames_dropped', 0) or 0)
        / max(1, (s.get('frames_decoded', 0) or 0) + (s.get('frames_dropped', 0) or 0))
        for s in t['stats']
    )
    for t in tiers.values()
) * 1.05
RES_YMAX = max(max(s.get('frame_height', 0) or 0 for s in t['stats']) for t in tiers.values()) * 1.1
FRAMES_YMAX = max(max(s.get('frames_decoded', 0) or 0 for s in t['stats']) for t in tiers.values()) * 1.05
BUFFER_YMAX = max(
    max(
        1000 * (s.get('jitter_buffer_delay_seconds', 0) or 0)
        / max(1, s.get('jitter_buffer_emitted_count', 0) or 1)
        for s in t['stats']
    )
    for t in tiers.values()
) * 1.1

summary_rows = []
for capacity_mbps, t in sorted(tiers.items()):
    s = t['stats']
    summary_rows.append({
        'Capacity (Mbps)': capacity_mbps,
        'Samples': len(s),
        'Final resolution': s[-1].get('resolution'),
        'Dropped frames (final)': s[-1].get('frames_dropped'),
        'Decoded frames (final)': s[-1].get('frames_decoded'),
        'New freezes': (s[-1].get('freeze_count', 0) or 0) - (s[0].get('freeze_count', 0) or 0),
    })
summary = pd.DataFrame(summary_rows).set_index('Capacity (Mbps)')
display(summary)


## Per-tier throughput

In [ ]:
CLIENT_IP = '172.16.1.1'

def classify_packets(pcap, window_start, window_end, direction_filter, remote_ip_field):
    output = subprocess.run(
        ['tshark', '-r', str(pcap), '-Y', direction_filter, '-T', 'fields',
         '-e', 'frame.time_epoch', '-e', remote_ip_field, '-e', 'frame.len'],
        check=True, capture_output=True, text=True,
    ).stdout
    rows = []
    for line in output.splitlines():
        fields = line.split('\t')
        if len(fields) < 3 or not fields[0] or not fields[1] or not fields[2]:
            continue
        timestamp = float(fields[0])
        if not window_start <= timestamp <= window_end:
            continue
        frame_bytes = int(fields[2].split(',')[0])
        rows.append((timestamp, frame_bytes))
    frame = pd.DataFrame(rows, columns=['timestamp', 'frame_bytes'])
    frame['second'] = (frame.timestamp - window_start).astype(int)
    return frame

for capacity_mbps, t in tiers.items():
    pcap = next(t['dir'].glob('*.pcap'))
    window_start = t['qoe'][0]['timestamp']
    window_end = t['qoe'][-1]['timestamp']
    window_seconds = window_end - window_start
    bin_count = max(1, math.ceil(window_seconds))

    download_packets = classify_packets(pcap, window_start, window_end, f'ip.dst == {CLIENT_IP}', 'ip.src')
    upload_packets = classify_packets(pcap, window_start, window_end, f'ip.src == {CLIENT_IP}', 'ip.dst')

    def per_second_mbps(packets):
        return (packets.groupby('second').frame_bytes.sum()
                .reindex(range(bin_count), fill_value=0)) * 8 / 1_000_000

    t['download_mbps'] = per_second_mbps(download_packets)
    t['upload_mbps'] = per_second_mbps(upload_packets)

DOWNLOAD_YMAX = max(max(CAPACITIES), max(t['download_mbps'].max() for t in tiers.values())) * 1.05
UPLOAD_YMAX = max(t['upload_mbps'].max() for t in tiers.values()) * 1.05

throughput_rows = []
for capacity_mbps, t in sorted(tiers.items()):
    throughput_rows.append({
        'Capacity (Mbps)': capacity_mbps,
        'Avg download (Mbps)': round(t['download_mbps'].mean(), 3),
        'Peak download (Mbps)': round(t['download_mbps'].max(), 3),
        'Avg upload (Mbps)': round(t['upload_mbps'].mean(), 3),
        'Peak upload (Mbps)': round(t['upload_mbps'].max(), 3),
    })
display(pd.DataFrame(throughput_rows).set_index('Capacity (Mbps)'))


## Download throughput

In [ ]:
for capacity_mbps in CAPACITIES:
    t = tiers[capacity_mbps]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
    ax.plot(t['download_mbps'].index, t['download_mbps'], marker='o', color=TEAMS_COLOR)
    ax.axhline(capacity_mbps, color='#333333', linestyle='--', alpha=.7, label=f'Configured bottleneck ({capacity_mbps} Mbps)')
    ax.set_ylim(0, DOWNLOAD_YMAX)
    ax.set(title=f'Teams Download Throughput — {capacity_mbps} Mbps', xlabel='Seconds from synchronized start', ylabel='Downloaded Mbit in each 1-second bin')
    ax.legend()
    plt.tight_layout()
    plt.show()


## Upload throughput

In [ ]:
for capacity_mbps in CAPACITIES:
    t = tiers[capacity_mbps]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
    ax.plot(t['upload_mbps'].index, t['upload_mbps'], marker='o', color=TEAMS_COLOR)
    ax.set_ylim(0, UPLOAD_YMAX)
    ax.set(title=f'Teams Upload Throughput — {capacity_mbps} Mbps', xlabel='Seconds from synchronized start', ylabel='Uploaded Mbit in each 1-second bin')
    plt.tight_layout()
    plt.show()


## Buffer health

In [ ]:
for capacity_mbps in CAPACITIES:
    t = tiers[capacity_mbps]
    jitter_ms = [
        1000 * (s.get('jitter_buffer_delay_seconds', 0) or 0)
        / max(1, s.get('jitter_buffer_emitted_count', 0) or 1)
        for s in t['stats']
    ]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
    ax.plot(t['seconds'], jitter_ms, marker='s', color=TEAMS_COLOR)
    ax.set_ylim(0, BUFFER_YMAX)
    ax.set(title=f'Teams Buffer Health — {capacity_mbps} Mbps', xlabel='Seconds from synchronized start', ylabel='Jitter buffer delay (ms, per emitted frame)')
    plt.tight_layout()
    plt.show()


## Dropped frame rate

In [ ]:
for capacity_mbps in CAPACITIES:
    t = tiers[capacity_mbps]
    drop_pct = [
        100 * (s.get('frames_dropped', 0) or 0)
        / max(1, (s.get('frames_decoded', 0) or 0) + (s.get('frames_dropped', 0) or 0))
        for s in t['stats']
    ]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
    ax.plot(t['seconds'], drop_pct, marker='o', color=TEAMS_COLOR)
    ax.set_ylim(0, DROP_YMAX)
    ax.set(title=f'Teams Cumulative Dropped Frame Rate — {capacity_mbps} Mbps', xlabel='Seconds from synchronized start', ylabel='Dropped frames (% of decoded frames so far)')
    plt.tight_layout()
    plt.show()


## Video resolution over time

In [ ]:
for capacity_mbps in CAPACITIES:
    t = tiers[capacity_mbps]
    heights = [s.get('frame_height', 0) or 0 for s in t['stats']]
    widths = [s.get('frame_width', 0) or 0 for s in t['stats']]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
    ax.step(t['seconds'], heights, where='post', color=TEAMS_COLOR, linewidth=2)
    ax.set_ylim(0, RES_YMAX)
    resolution_labels = {}
    for width, height in zip(widths, heights):
        if width and height:
            resolution_labels.setdefault(height, set()).add(f'{width}x{height}')
    tick_heights = sorted(resolution_labels)
    if tick_heights:
        ax.set_yticks(tick_heights, [' / '.join(sorted(resolution_labels[h])) for h in tick_heights])
    ax.set(title=f'Teams Video Resolution Over Time — {capacity_mbps} Mbps', xlabel='Seconds from synchronized start', ylabel='Rendered/decoded resolution')
    plt.tight_layout()
    plt.show()


## Total video frames

In [ ]:
for capacity_mbps in CAPACITIES:
    t = tiers[capacity_mbps]
    frames = [s.get('frames_decoded', 0) or 0 for s in t['stats']]
    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
    ax.plot(t['seconds'], frames, marker='o', color=TEAMS_COLOR)
    ax.set_ylim(0, FRAMES_YMAX)
    ax.set(title=f'Teams Total Video Frames — {capacity_mbps} Mbps', xlabel='Seconds from synchronized start', ylabel='Cumulative decoded video frames')
    plt.tight_layout()
    plt.show()
